[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/pytorch-lab/blob/main/02_neural_net_intro.ipynb)

# 02 - Введение в нейронные сети

**Цель.** Связать тензоры с обучаемой моделью: построить один нейрон, обучить его на двумерных данных и визуализировать границу решения.

**Результаты обучения:**
- объяснять формулу нейрона y = f(w·x + b)
- создавать синтетический датасет классификации
- задавать один нейрон через nn.Linear
- писать ручной цикл обучения
- сравнивать sigmoid, ReLU и tanh

## Источники
- [Heaton: Neural network intro](https://github.com/jeffheaton/app_deep_learning/blob/main/t81_558_class_01_3_neural_net.ipynb)

Импортируем библиотеки и фиксируем случайные зерна. Для визуализации будем использовать matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import make_blobs
from torch import nn

%matplotlib inline

torch.manual_seed(42)
np.random.seed(42)

## Краткая теория

Один искусственный нейрон вычисляет выражение `y = f(w·x + b)`, где `x` - входные признаки, `w` - обучаемые веса, `b` - смещение, а `f` - функция активации. Sigmoid переводит значение в диапазон от 0 до 1, ReLU зануляет отрицательные значения, tanh возвращает значения от -1 до 1.

Обучение означает подбор `w` и `b`, при котором функция потерь становится меньше.

Создадим два линейно разделимых облака точек. Цвет точки соответствует классу.

In [ ]:
X, y = make_blobs(n_samples=200, centers=2, random_state=42, cluster_std=1.5)
X = X.astype(np.float32)
y = y.astype(np.float32).reshape(-1, 1)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="coolwarm", edgecolor="k")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Synthetic binary classification data")
plt.show()

Опишем модель как линейный слой с двумя входами и одним выходом, после которого применяется sigmoid. Сначала посмотрим на необученную границу решения.

In [ ]:
X_tensor = torch.from_numpy(X)
y_tensor = torch.from_numpy(y)

model = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())

def plot_decision_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    with torch.no_grad():
        probs = model(grid).reshape(xx.shape).numpy()

    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, probs, levels=20, cmap="coolwarm", alpha=0.35)
    plt.contour(xx, yy, probs, levels=[0.5], colors="black")
    plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="coolwarm", edgecolor="k")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(title)
    plt.show()

plot_decision_boundary(model, X, y, "Initial decision boundary")

Запустим ручной цикл обучения: прямой проход, расчет `BCELoss`, обратное распространение и шаг `SGD`.

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
loss_history = []

for epoch in range(200):
    optimizer.zero_grad()
    predictions = model(X_tensor)
    loss = criterion(predictions, y_tensor)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())

print(f"Final loss: {loss_history[-1]:.4f}")

После обучения граница решения должна пройти между двумя классами. *

In [ ]:
plot_decision_boundary(model, X, y, "Trained decision boundary")

plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training loss")
plt.show()

Сравним формы трех популярных функций активации на одном графике. *

In [ ]:
x_values = torch.linspace(-5, 5, 300)
sigmoid = torch.sigmoid(x_values)
relu = torch.relu(x_values)
tanh = torch.tanh(x_values)

plt.figure(figsize=(7, 4))
plt.plot(x_values, sigmoid, label="sigmoid")
plt.plot(x_values, relu, label="ReLU")
plt.plot(x_values, tanh, label="tanh")
plt.xlabel("Input")
plt.ylabel("Output")
plt.title("Activation functions")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

> Материал основан на курсе Jeff Heaton T81-558 и книге Maxim Lapan 'Deep Reinforcement Learning Hands-On', глава 3. Адаптировано для учебных целей.